# Dashboard Tabs 1 & 2
_by Juan Mamani-Rojas_ _([LinkedIn](https://linkedin.com/in/jjvmr))_
## Data processing
### Preparing data for visualization

First import the necessary libraries to process your data and have it ready to use in IBM Cognos, Google Looker, or any other visualization tool.
_Note: You can always create your visualization within the jupyter notebook._


In [1]:
import pandas as pd
import numpy as np
import os

### Create df of data obtained

In [2]:
df_original = pd.read_csv("D:/_JJProjects/JupyterProject/survey_data.csv")

# Set up list
By creating a dictionary of keys:values, we are able to map through the list and create different variables and names for our different datasets.

In [3]:
# Make a list of columns and variable_names that correespond to the column to create datasets ready for visualizations
columns_to_process = {
    'LanguageHaveWorkedWith': 'lgexp',
    'LanguageWantToWorkWith': 'lgwant',
    'DatabaseHaveWorkedWith': 'dbexp',
    'DatabaseWantToWorkWith': 'dbwant',
    'PlatformHaveWorkedWith': 'platexp',
    'PlatformWantToWorkWith': 'platwant',
    'WebframeHaveWorkedWith': 'wfexp',
    'WebframeWantToWorkWith': 'wfwant'
}

# Create a directory for outputs if it doesn't exist
output_dir = 'processed_dfs'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Pre-processing and cleaning of Top 10 trends

## Brief summary
The following code iterates through a predefined list of column mappings (columns_to_process), performing a series of data manipulation steps within a single, efficient loop.
### 1. Splitting data
For each column, it isolates the ResponseId and the target technology column and applies a pandas ```explode()``` operation, transforming the list of technologies into individual rows.

### 2. Exploding process into different rows but same ResponseId per response.
Immediately following the explode operation, the code cleans up any leading or trailing whitespace from the technology names using ```str.strip()``` and then removes duplicate ```[ResponseId, TechName]``` combinations to ensure each respondent is counted only once per unique skill.

### 3. Aggregation and filtering (saving data)
the script performs aggregation: it groups the cleaned data by the technology name, counts the number of unique ResponseId entries, sorts these counts in descending order, and selects the top 10 results.\
 It filters the original unique dataframe to keep only these top 10 entries and saves the resulting, highly focused dataframe to a CSV file in a subdirectory named ```processed_dfs```, using a dynamic file name (e.g., ```havedb_top10.csv```) generated from the ```new_var_name``` variable in the loop.

In [4]:
# Iterate over the mapping defined above
for original_col_name, new_var_name in columns_to_process.items():
    if original_col_name in df_original.columns:

        # 1. Ensure the column is split by semicolon first (this typically happens outside the main loop,
        #    but we can put it here if we assume it hasn't happened yet for safety)
        if df_original[original_col_name].dtype == 'object' and df_original[original_col_name].str.contains(';').any():
             df_original[original_col_name] = df_original[original_col_name].str.split(';')

        # 2. Explode the data into a temporary dataframe
        exploded_df = df_original[['ResponseId', original_col_name]].explode(original_col_name)

        # 3. Clean up whitespaces using the dynamic column name
        exploded_df[original_col_name] = exploded_df[original_col_name].str.strip()

        # 4. Drop duplicates
        unique_df = exploded_df.drop_duplicates(subset=['ResponseId', original_col_name])

        # 5. Count unique respondents per category
        counts = (
            unique_df.groupby(original_col_name)['ResponseId']
            .nunique()
            .sort_values(ascending=False)
        )

        # 6. Keep top 10
        top10 = counts.head(10)

        # 7. Filter the unique_df to only include the top 10 categories
        top10_df = unique_df[unique_df[original_col_name].isin(top10.index)]

        # 8. Save the resulting top10_df to a file
        filename = f"{new_var_name}_top10.csv"
        filepath = os.path.join(output_dir, filename)

        top10_df.to_csv(filepath, index=False)

        print(f"Saved {filepath} with top 10 {original_col_name} data.")

Saved processed_dfs\lgexp_top10.csv with top 10 LanguageHaveWorkedWith data.
Saved processed_dfs\lgwant_top10.csv with top 10 LanguageWantToWorkWith data.
Saved processed_dfs\dbexp_top10.csv with top 10 DatabaseHaveWorkedWith data.
Saved processed_dfs\dbwant_top10.csv with top 10 DatabaseWantToWorkWith data.
Saved processed_dfs\platexp_top10.csv with top 10 PlatformHaveWorkedWith data.
Saved processed_dfs\platwant_top10.csv with top 10 PlatformWantToWorkWith data.
Saved processed_dfs\wfexp_top10.csv with top 10 WebframeHaveWorkedWith data.
Saved processed_dfs\wfwant_top10.csv with top 10 WebframeWantToWorkWith data.


### That's all.
The files were downloaded within the directory where the python script was ran.\
Simply upload these datasets into any visualization tool and it will be ready to create any charts.
# Thank you!